## Allocation Algorithms for Full-scale instance

This notebook walks through the steps needed to generate a course schedule using real Fall 2024 class data and real student preferences.
For testing, the notebook uses the synthetic dataset (`survey_data.csv`), which has roughly 700 responses. Then, it builds a scaled instance to match real students per status proportion, and adjusts the course capacities accordingly. 

The code relies on the **course-data** repository to parse and build the student objects, and on the **yankee-swap-framework** repository to run the allocation algorithms. After generating allocations, the notebook evaluates them using several metrics, including:

- Utilitarian Social Welfare (USW)  
- Nash Social Welfare (NSW)  
- Envy

The algorithms are implemented assuming **binary submodular valuation functions**, which aligns with the valuation structure required for Yankee Swap. However, they can also work with the actual valuations from the survey (the scores students assigned to each class).


#### Imports

In [ ]:
import numpy as np
import random
import copy 

from fair.stats.survey import Corpus, SingleTopicSurvey
from fair.agent import LegacyStudent
from fair.allocation import general_yankee_swap_E, round_robin, serial_dictatorship, integer_linear_program
from fair.optimization import StudentAllocationProgram
from fair.metrics import utilitarian_welfare, nash_welfare
from fair.envy import EF_violations_reponses, EF_violations
from matplotlib import pyplot as plt
from sklearn.decomposition import PCA

import qsurvey

#### Define Parameters

In [ ]:
status_color_map = {
    1: "lightsteelblue",
    2: "blue",
    3: "forestgreen",
    4: "darkkhaki",
    5: "darkorange",
    6: "red",
}
status_max_course_map = {
    1: 6,
    2: 6,
    3: 6,
    4: 6,
    5: 4,
    6: 4,
}
status_crs_prefix_map = {
    1: ["1", "2", "3"],
    2: ["1", "2", "3", "4"],
    3: ["1", "2", "3", "4", "5"],
    4: ["2", "3", "4", "5", "6"],
    5: ["5", "6"],
    6: ["5", "6"],
}
NUM_STUDENTS_PER_STATUS = {
    1: 239,
    2: 327,
    3: 408,
    4: 573,
    5: 613,
    6: 148,
}

SPARSE = False
pref_thresh = 10
seed = 43

#### Generate schedule and students

Read survey data form `survey_data.csv`, which consists of roughly ~700 student real responses. 
Generating students make take several minutes (8-10 mins).
Course capacities are scaled to roughly match the instance size.

In [ ]:

survey_file = "../resources/survey_data.csv"
schedule_file = "../resources/anonymized_courses.xlsx"
mapping_file = "../resources/survey_column_mapping.csv"

mp = qsurvey.QMapper(mapping_file)
qd = qsurvey.QSchedule(schedule_file)
crs_sec_cap_map = qd.capacities()
qs = qsurvey.QSurvey(survey_file, mp, list(crs_sec_cap_map.keys()))
course_map = mp.mapping(qs.all_courses)
all_courses = [crs for crs in course_map.keys()]
features = mp.features(course_map)
course, slot, weekday, section = features
schedule = mp.schedule(course_map, crs_sec_cap_map, features)
students, responses, statuses = qs.students(
    course_map, all_courses, features, schedule, status_max_course_map, pref_thresh, SPARSE
)
student_status_map = {students[i]: status for i, status in enumerate(statuses)}
student_resp_map = {students[i]: response for i, response in enumerate(responses)}
course_cap_map = {
    crs: crs_sec_cap_map[course_map[crs]["course num"]][int(course_map[crs]["section"])]
    for crs in all_courses
}
all_students = [
    student for student in students if len(student.student.preferred_courses) > 0
]

n_responses_per_status = np.zeros(6)
for student in all_students:
    student_status = int(student_status_map[student])
    n_responses_per_status[student_status - 1] += 1

rates = [n_responses_per_status[i] / NUM_STUDENTS_PER_STATUS[i + 1] for i in range(6)]
rate = min(rates)

n_per_status = [round(NUM_STUDENTS_PER_STATUS[i + 1] * min(rates)) for i in range(6)]

# Reduce course capacities to match student scale 
for sche in schedule:
    sche.capacity = round(sche.capacity * rate)

# Randomly select students from different status to match scale
random.seed(seed)
reduced_students = []
for status in range(1, 7):
    students_status = [
        student for student in all_students if student_status_map[student] == status
    ]
    selected_students = random.sample(students_status, n_per_status[status - 1])
    reduced_students = [*reduced_students, *selected_students]
students.sort(key=lambda x: student_status_map[x])
students.reverse()
students = reduced_students
print("Num students,", len(students))

#### Allocation Algorithms: Binary submodular scenario

With the built instance (students and schedule), we can run each of the allocation algorithms, and evaluate their performance according to USW, NSW and Envy metrics. Envy metric is commented out as it takes a while to compute (>5 minutes).

In [ ]:
X_ILP = integer_linear_program(students, schedule)
print(f"Integer Linear Program USW:{utilitarian_welfare(X_ILP, students, schedule)}")
print(f"Integer Linear Program NSW:{nash_welfare(X_ILP, students, schedule)}")
# print(f"Integer Linear Program EF violations:{EF_violations(X_ILP, students, schedule)}")

In [ ]:
X_SD = serial_dictatorship(students, schedule)
print(f"Serial Dictatorship USW:{utilitarian_welfare(X_SD, students, schedule)}")
print(f"Serial Dictatorship NSW:{nash_welfare(X_SD, students, schedule)}")
# print(f"Serial Dictatorship EF violations:{EF_violations(X_SD, students, schedule)}")

In [ ]:
X_RR = round_robin(students, schedule)
print(f"Round Robin USW:{utilitarian_welfare(X_SD, students, schedule)}")
print(f"Round Robin NSW:{nash_welfare(X_SD, students, schedule)}")
# print(f"Round Robin EF violations:{EF_violations(X_SD, students, schedule)}")

In [ ]:
X_YS,_ ,_ = general_yankee_swap_E(students, schedule)
print(f"YS USW:{utilitarian_welfare(X_YS, students, schedule)}")
print(f"YS NSW:{nash_welfare(X_YS, students, schedule)}")
# print(f"YS EF violations:{EF_violations(X_YS, students, schedule)}")

#### Allocation Algorithms: Additive scenario

With the same instance, we run the algorithms again, but now considering that students have additive valuation functions over the items, given by their survey responses. In this case, ILP, Serial Dictatorship and Round Robin run with additive utilities, while Yankee Swap runs considering binary submodular valuations, and taking into account the response values only for tie-breaking purposes. Again, Envy metric is commented out as it takes a while to compute (>5 minutes).

In [ ]:
# Define the response values for each student
c = np.vstack([student_resp_map[student] for student in students])-1

In [ ]:
X_ILP = integer_linear_program(students, schedule, valuations=c)
current_utilities = np.diag(np.dot(c,X_ILP))
print(f"Integer Linear Program USW: {utilitarian_welfare(X_ILP, students, schedule, current_utilities)}")
print(f"Integer Linear Program NSW: {nash_welfare(X_ILP, students, schedule, current_utilities)}")
# EF = EF_violations_reponses(X_ILP, students, schedule, student_status_map,c)
# print(f"Integer Linear Program EF violations: {EF[0]}")
# print(f"Integer Linear Program downwards EF violations: {EF[1]}")

In [ ]:
X_SD = serial_dictatorship(students, schedule, valuations=c)
current_utilities = np.diag(np.dot(c,X_SD))
print(f"Serial Dictatorship USW: {utilitarian_welfare(X_SD, students, schedule, current_utilities)}")
print(f"Serial Dictatorship NSW: {nash_welfare(X_SD, students, schedule, current_utilities)}")
# EF = EF_violations_reponses(X_SD, students, schedule, student_status_map,c)
# print(f"Serial Dictatorship EF violations: {EF[0]}")
# print(f"Serial Dictatorship downwards EF violations: {EF[1]}")

In [ ]:
X_RR = round_robin(students, schedule, valuations=c)
current_utilities = np.diag(np.dot(c,X_RR))
print(f"Round Robin USW: {utilitarian_welfare(X_RR, students, schedule, current_utilities)}")
print(f"Round Robin NSW: {nash_welfare(X_RR, students, schedule, current_utilities)}")
# EF = EF_violations_reponses(X_RR, students, schedule, student_status_map,c)
# print(f"Round Robin EF violations: {EF[0]}")
# print(f"Round Robin downwards EF violations: {EF[1]}")

In [ ]:
X_YS, _, _ = general_yankee_swap_E(students, schedule, valuations=c)
current_utilities = np.diag(np.dot(c,X_YS))
print(f"Yankee Swap USW: {utilitarian_welfare(X_YS, students, schedule, current_utilities)}")
print(f"Yankee Swap NSW: {nash_welfare(X_YS, students, schedule, current_utilities)}")
# EF = EF_violations_reponses(X_YS, students, schedule, student_status_map,c)
# print(f"Yankee Swap EF violations: {EF[0]}")
# print(f"Yankee Swap downwards EF violations: {EF[1]}")